# IFUsersFlows with GC1DPP Integration

This notebook demonstrates a collaborative bike production workflow using ValueFlows (via Zenflows) integrated with the Global Circularity 1 Digital Product Passport (GC1DPP) standard.

## Workflow Overview

The workflow follows this sequence:

1. **Setup**: Initialize users, locations, units, and resource specifications
2. **Component Production**: Create bike components (frame, mirror, design)
3. **DPP Creation**: Create and submit a GC1DPP **BEFORE** producing the final bike
4. **Bike Production**: Produce the collaborative bike with the DPP ULID embedded in its metadata
5. **Tracing & Visualization**: Query and visualize the complete supply chain

## Key Innovation: DPP-First Workflow

Unlike traditional workflows where documentation is added after production, this notebook implements a **DPP-first** approach:
- The GC1DPP is created and submitted **before** the final bike is produced
- The DPP's ULID is embedded in the bike's metadata at creation time
- This ensures traceability is baked into the product from the moment of creation

This matches the frontend pattern used in Interfacer where the DPP is created first, then its ULID is included in the project creation mutation.

In [52]:
# The following should reload if_lib if we change it on disk
# %load_ext autoreload
%reload_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location

from if_dpp import trace_query, check_traces, er_before, get_dpp

from if_graphics import vis_dpp, make_sankey, consol_trace

from if_gc1dpp import submit_dpp, create_sample_bike_dpp

## 1. Module Imports and Auto-Reload Setup

This cell sets up automatic module reloading and imports all necessary libraries:

- **if_lib**: Core Zenflows/ValueFlows interaction functions
- **if_gc1dpp**: GC1DPP (Global Circularity 1 Digital Product Passport) integration
- **if_dpp**: Digital Product Passport tracing and querying
- **if_graphics**: Visualization tools for supply chain flows
- **if_utils**: Utility functions for data management

The `%autoreload` magic ensures that if we modify any of the imported modules during development, they will be automatically reloaded.

In [53]:
# We define the constant for our case
USE_CASE = 'ifusersflows'

# What endpoint are we talking to?
# debug
# ENDPOINT = 'http://65.109.11.42:10000/api'
# ENDPOINT = 'http://zenflows-debug.interfacer.dyne.org/api'
# staging
# ENDPOINT = 'http://65.109.11.42:8000/api'
# ENDPOINT = 'https://zenflows-staging.interfacer.dyne.org/api'
# ENDPOINT = 'https://zenflows.interfacer-staging.dyne.im/api'
# testing
# ENDPOINT = 'http://65.109.11.42:9000/api'
# ENDPOINT = 'https://zenflows-test.interfacer.dyne.org/api'
ENDPOINT = 'https://proxy.dpp-test.dyne.im/zenflows/api'

# DPP service endpoint
DPP_URL = 'http://localhost:8080'

USERS = ['designer1', 'designer2', 'service_prov1', 'service_prov2', 'manufacturer1', 'manufacturer2', 'customer1', 'customer2']

## 2. Configuration and Endpoints

Define the endpoints and use case identifier:

- **ENDPOINT**: The Zenflows API endpoint (ValueFlows implementation)
- **DPP_URL**: The GC1DPP service endpoint for storing Digital Product Passports
- **USE_CASE**: Identifier for this workflow ('ifusersflows')
- **USERS**: List of all participants in the supply chain

The workflow involves multiple actors:
- **Designers** (designer1, designer2): Create designs and assemble the bike
- **Manufacturers** (manufacturer1, manufacturer2): Produce physical components
- **Service Providers** (service_prov1, service_prov2): Provide manufacturing/assembly services
- **Customers** (customer1, customer2): End users

In [54]:
# Calculate names of settings files
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)

## 3. File Path Configuration

Generate file paths for persistent data storage. Files are organized by endpoint and use case to support multiple environments (staging, production, test).

Files stored:
- **cred_users.json**: User credentials, keys, and IDs
- **loc_users.json**: Geographic locations for all users
- **units_data.json**: Measurement units (kg, pieces, hours, etc.)
- **res_spec_data.json**: Resource specifications (types of resources)
- **dpp_data.json**: GC1DPP records linked to resources

In [55]:
# Read or define user data that is going to be used in the GraphQL calls

# create data structure to hold processes
process_data = {}

# create data structures to hold resources and events (possibly to compare results from track and trace)
res_data = {}
event_seq = []

# create data structure to hold DPP data
dpp_data = {}

if os.path.isfile(USERS_FILE):
    with open(USERS_FILE,'r') as f:
        users_data = json.loads(f.read())
    print("Credentials file available for users")
else:
    users_data = {}
    users_data['designer1'] = {
      "userChallenges": {
        "whereParentsMet": "London",
        "nameFirstPet": "Fuffy",
        "nameFirstTeacher": "Jim",
        "whereHomeTown": "Paris",
        "nameMotherMaid": "Wright"
      },
      "name": "Designer1",
      "username": "designer1_username",
      "email": "designer1@example.org",
      "note": "me.designer1.org"
    }
    users_data['designer2'] = {
      "userChallenges": {
        "whereParentsMet": "London",
        "nameFirstPet": "Fido",
        "nameFirstTeacher": "Mary",
        "whereHomeTown": "Amsterdam",
        "nameMotherMaid": "Wraight"
      },
      "name": "Designer2",
      "username": "designer2_username",
      "email": "designer2@example.org",
      "note": "me.designer2.org"
    }

    users_data['manufacturer1'] = {
        "userChallenges": {
            "whereParentsMet":"Amsterdam",
            "nameFirstPet":"Toby",
            "nameFirstTeacher":"Juliet",
            "whereHomeTown":"Rome",
            "nameMotherMaid":"Banks"
        },
        "name": "Manufacturer1",
        "username": "manufacturer1_username",
        "email": "manufacturer1@example.org",
        "note" : "me.manufacturer1.org"
    }
    users_data['manufacturer2'] = {
        "userChallenges": {
            "whereParentsMet":"Canicatti",
            "nameFirstPet":"Ula",
            "nameFirstTeacher":"Pugsly",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Ranks"
        },
        "name": "Manufacturer2",
        "username": "manufacturer2_username",
        "email": "manufacturer2@example.org",
        "note" : "me.manufacturer2.org"
    }
    users_data['service_prov1'] = {
        "userChallenges": {
            "whereParentsMet":"Amsterdam",
            "nameFirstPet":"Toby",
            "nameFirstTeacher":"Juliet",
            "whereHomeTown":"Rome",
            "nameMotherMaid":"Banks"
        },
        "name": "Service_prov1",
        "username": "service_prov1_username",
        "email": "service_prov1@example.org",
        "note" : "me.service_prov1.com"
    }
    users_data['service_prov2'] = {
        "userChallenges": {
            "whereParentsMet":"Canicatti",
            "nameFirstPet":"Ula",
            "nameFirstTeacher":"Pugsly",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Ranks"
        },
        "name": "service_prov2",
        "username": "service_prov2_username",
        "email": "service_prov2@example.org",
        "note" : "me.service_prov2.com"
    }


    users_data['customer1'] = {
        "userChallenges": {
            "whereParentsMet":"Rome",
            "nameFirstPet":"Ku",
            "nameFirstTeacher":"George",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Canti"
        },
        "name": "Customer1",
        "username": "customer1_username",
        "email": "customer1@example.org",
        "note" : "me.customer1.org"
    }

    users_data['customer2'] = {
        "userChallenges": {
            "whereParentsMet":"Rome",
            "nameFirstPet":"Ku",
            "nameFirstTeacher":"George",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Canti"
        },
        "name": "Customer2",
        "username": "customer2_username",
        "email": "customer2@example.org",
        "note" : "me.customer2.org"
    }

    with open(USERS_FILE,'w') as f:
        json.dump(users_data, f)

if os.path.isfile(LOCS_FILE):
    with open(LOCS_FILE,'r') as f:
        locs_data = json.loads(f.read())
    print("Location file available")
else:
    locs_data = {}
    locs_data['designer1'] = {
        "name": "Dyne",
        "lat": 52.39679,
        "long": 4.8781073,
        "addr": "Haparandadam 7, A1, 1013 AK Amsterdam, Netherlands",
        "note": "location.designer1.org"
    }
    locs_data['designer2'] = {
        "name": "Farback",
        "lat": 52.3767127,
        "long": 4.8990591,
        "addr": "Prins Hendrikkade 82 A, 1012 AE, Amsterdam, Netherlands",
        "note": "location.designer2.org"
    }

    locs_data['manufacturer1'] = {
        "name": "Fab Lab Hamburg",
        "lat" : 51.1531305,
        "long" : 5.2685045,
        "addr" : "Stockmeyerstr. 43 Halle 4K Eingang von der Wasserseite, 20457 Hamburg, Germany",
        "note": "location.manufacturer1.org"
    }

    locs_data['manufacturer2'] = {
        "name": "Fab Lab Amsterdam",
        "lat" : 52.372773,
        "long" : 4.8981243,
        "addr" : "Nieuwmarkt 4, 1012 CR Amsterdam, Netherlands",
        "note": "location.manufacturer2.org"
    }

    locs_data['service_prov1'] = {
        "name": "Bike Totaal B.V.",
        "lat" : 52.1971579,
        "long" : 5.3924199,
        "addr" : "Spaceshuttle 22, 3824 ML Amersfoort, Netherlands",
        "note": "location.service_prov1.com"
    }

    locs_data['service_prov2'] = {
        "name": "Lyondell Covestro Manufacturing",
        "lat" : 51.9650382,
        "long" : 4.0133683,
        "addr" : "Australiëweg 7, 3199 KB Maasvlakte Rotterdam, Netherlands",
        "note": "location.service_prov2.com"
    }

    locs_data['customer1'] = {
        "name": "CleanLease",
        "lat" : 51.47240440868687,
        "long" : 5.412460440524406,
        "addr" : "De schakel 30, 5651 Eindhoven, Netherlands",
        "note": "location.customer1.org"
    }

    locs_data['customer2'] = {
        "name": "OLVG",
        "lat" : 52.3585703,
        "long" : 4.9124307,
        "addr" : "Oosterpark 9, 1091 AC Amsterdam, Netherlands",
        "note": "location.customer2.org"
    }

    with open(LOCS_FILE,'w') as f:
        json.dump(locs_data, f)

if os.path.isfile(UNITS_FILE):
    with open(UNITS_FILE,'r') as f:
        units_data = json.loads(f.read())
    print(f"Unit file available")
else:
    units_data = {}


if os.path.isfile(SPECS_FILE):
    with open(SPECS_FILE,'r') as f:
        res_spec_data = json.loads(f.read())
    print(f"Resource Spec file available")
else:
    res_spec_data = {}

Credentials file available for users
Location file available
Unit file available
Resource Spec file available


## 4. Initialize User and Location Data

Load or create user profiles and location data:

**User Data** includes:
- Identity information (name, username, email)
- Security challenges (for key generation using Zenroom)
- Cryptographic keys (generated later)
- Agent IDs in the Zenflows system

**Location Data** includes:
- Geographic coordinates (lat/long)
- Physical addresses
- Location identifiers in the Zenflows system

Data is either loaded from existing files or initialized with default values for first-time setup.

In [56]:
# Read HMAC or get it from the server
for user in USERS:
    read_HMAC(USERS_FILE, users_data, user, endpoint=ENDPOINT)

Server HMAC available for Designer1
Server HMAC available for Designer2
Server HMAC available for Service_prov1
Server HMAC available for service_prov2
Server HMAC available for Manufacturer1
Server HMAC available for Manufacturer2
Server HMAC available for Customer1
Server HMAC available for Customer2


## 5. Authentication Setup: HMAC Generation

Request HMAC (Hash-based Message Authentication Code) from the Zenflows server for each user.

The HMAC is used as a server-side seed component for cryptographic key generation. Combined with the user's challenge answers, it enables deterministic key generation while maintaining security.

In [57]:
# Read the keypair
for user in USERS:
    read_keypair(USERS_FILE, users_data, user)

Keypair available for Designer1
Keypair available for Designer2
Keypair available for Service_prov1
Keypair available for service_prov2
Keypair available for Manufacturer1
Keypair available for Manufacturer2
Keypair available for Customer1
Keypair available for Customer2


## 6. Cryptographic Key Generation

Generate EdDSA keypairs for each user using Zenroom cryptographic library.

Keys are derived from:
- User challenge answers (security questions)
- Server-provided HMAC

This creates a **deterministic but secure** keypair that can be regenerated if needed, without storing private keys. The EdDSA public key is used for authentication with Zenflows and for signing DPP submissions.

In [58]:
# read or get id of the person
for user in USERS:
    get_id_person(USERS_FILE, users_data, user, endpoint=ENDPOINT)

Id available for Designer1
Id available for Designer2
Id available for Service_prov1
Id available for service_prov2
Id available for Manufacturer1
Id available for Manufacturer2
Id available for Customer1
Id available for Customer2


## 7. User Registration in Zenflows

Create or retrieve user (agent) IDs in the Zenflows system.

Each user is registered as an agent with:
- Public key (for cryptographic verification)
- Profile information (name, email)
- Unique agent ID

The agent ID is used in all subsequent ValueFlows transactions to identify who performs actions.

In [59]:
# Read of get the location id
for user in USERS:
    get_location_id(LOCS_FILE, users_data[user], locs_data, user, endpoint=ENDPOINT)
    set_user_location(USERS_FILE, users_data, locs_data, user, endpoint=ENDPOINT)

Location id available for Dyne
Location id available for Designer1
Location id available for Farback
Location id available for Designer2
Location id available for Bike Totaal B.V.
Location id available for Service_prov1
Location id available for Lyondell Covestro Manufacturing
Location id available for service_prov2
Location id available for Fab Lab Hamburg
Location id available for Manufacturer1
Location id available for Fab Lab Amsterdam
Location id available for Manufacturer2
Location id available for CleanLease
Location id available for Customer1
Location id available for OLVG
Location id available for Customer2


## 8. Location Registration and Assignment

Register geographic locations in Zenflows and associate them with users.

Two steps:
1. **get_location_id**: Creates a spatial entity in Zenflows with coordinates and address
2. **set_user_location**: Links the location as the user's primary location

Locations are important for tracking where resources are created and transferred.

In [60]:
# Get the ids of all units
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'piece', 'u_piece', 'om2:one', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'mass', 'kg', 'om2:kilogram', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'volume', 'lt', 'om2:litre', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'time', 'h', 'om2:hour', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'distance', 'km', 'om2:kilometer', endpoint=ENDPOINT)

Unit piece available
Unit mass available
Unit volume available
Unit time available
Unit distance available


## 9. Unit of Measurement Registration

Register standard units of measurement in Zenflows.

Units defined:
- **piece** (u_piece): Countable items (bikes, components)
- **mass** (kg): Weight measurements (aluminium, glass)
- **volume** (lt): Volume measurements
- **time** (h): Work hours (design, manufacturing effort)
- **distance** (km): Distance measurements

These units are referenced in all resource specifications and event quantities.

In [61]:
# Create the processes

# Create the process that delivers the service frame manufacturing
process_name = 'Create_bike_frame_manifacturing_service_ABC'
user_data = users_data['service_prov1']
note = f"Creation of bike frame manufacturing service ABC by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that delivers bike assembly service
process_name = 'Create_bike_assembly_service'
user_data = users_data['service_prov2']
note = f"Creation of bike assembly 1-hour service by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that deliver the service frame manufacturing
process_name = 'Create_aluminium_bike_frame'
user_data = users_data['manufacturer1']
note = f"Creation of aluminum bike frame by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating design
process_name = 'Create_funky_bike_design'
user_data = users_data['designer1']
note = f"Creation of funky bike design by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating the mirror
process_name = 'Creation_magic_bike_mirror'
user_data = users_data['manufacturer2']
note = f"Creation of magic bike mirror by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating the bike
process_name = 'Creation_fancy_collaborative_bike'
user_data = users_data['designer2']
note = f"Creation of fancy collaborative bike by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

## 10. Process Definition

Create ValueFlows processes that will contain economic events.

Processes represent transformations or activities:
- **Create_aluminium_bike_frame**: Manufacturing process for the frame
- **Creation_magic_bike_mirror**: Production of the mirror component
- **Create_funky_bike_design**: Design work process
- **Creation_fancy_collaborative_bike**: Final bike assembly process

Each process groups related events (consume inputs, produce outputs, cite references).

In [62]:
# Read all the resource specifications
name = 'bike_manufacturing_service'
note = 'Specification bike assembly service'
classification = 'https://www.service_prov1.com/manufacture/bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'aluminium'
note = 'Aluminium 10mt, 4cm pipes'
classification = 'https://www.wikidata.org/wiki/Q663'
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'aluminium_bike_frame'
note = 'Repository for aluminium bike frame'
classification = 'https://github.com/manufacturer1/aluminium_bike_frame'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'funky_bike_design'
note = 'Repository for funky bike design'
classification = 'https://github.com/designer1/funky_bike_design'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'glass'
note = 'Glass for magic bike mirror'
classification = 'https://www.wikidata.org/wiki/Q11469'
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'magic_bike_mirror'
note = 'Repository for magic bike mirror'
classification = 'https://github.com/manufacturer2/magic_bike_mirror'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'bike_assembly_service'
note = 'Specification bike assembly service'
classification = 'https://www.service_prov2.com/assembly/bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'fancy_collaborative_bike'
note = 'Repository for fancy collaborative bike'
classification = 'https://github.com/designer2/fancy_collaborative_bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'manufacturing_work'
note = 'Specification for manufacturing work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'assemblying_work'
note = 'Specification for assemblying work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'design_work'
note = 'Specification for design work'
classification = 'https://www.wikidata.org/wiki/Q82604'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

Specification bike_manufacturing_service available
Specification aluminium available
Specification aluminium_bike_frame available
Specification funky_bike_design available
Specification glass available
Specification magic_bike_mirror available
Specification bike_assembly_service available
Specification fancy_collaborative_bike available
Specification manufacturing_work available
Specification assemblying_work available
Specification design_work available


## 11. Resource Specification Registration

Define the types of resources that can exist in the system.

Resource specifications include:
- **Materials**: aluminium, glass (raw materials)
- **Components**: aluminium_bike_frame, magic_bike_mirror, funky_bike_design
- **Final Product**: fancy_collaborative_bike
- **Work Types**: manufacturing_work, assemblying_work, design_work
- **Services**: bike_manufacturing_service, bike_assembly_service

Each specification has:
- Name and description
- Classification (URL to standard ontology or schema)
- Default unit of measurement

In [63]:
# We create the resources that will not be saved to file as it is assumed they are recreated at each run

res_name = 'aluminium'
amount = 5
get_resource(res_data, res_spec_data, res_name, users_data['manufacturer1'], event_seq, amount, endpoint=ENDPOINT)

res_name = 'glass'
amount = .2
get_resource(res_data, res_spec_data, res_name, users_data['manufacturer2'], event_seq, amount, endpoint=ENDPOINT)

## 12. Initial Resource Creation

Create the raw material resources needed for component production:

- **5 kg of aluminium** (owned by manufacturer1) - will be used for the bike frame
- **0.2 kg of glass** (owned by manufacturer2) - will be used for the mirror

These resources are created using the 'raise' action (bringing resources into existence). Each resource gets a unique ID in the Zenflows system.

## 13. Component Production Flow

The following cells create the bike components through a series of ValueFlows economic events.

### Production Steps:
1. **Aluminium Bike Frame** (manufacturer1):
   - Consume 5 kg aluminium → Produce 1 bike frame
   - Transfer frame to designer2

2. **Magic Bike Mirror** (manufacturer2):
   - Consume 0.2 kg glass → Produce 1 mirror
   - Transfer mirror to designer2

3. **Funky Bike Design** (designer1):
   - Perform 16 hours of design work → Produce 1 design
   - Design stays with designer1 (will be cited, not consumed)

Each production involves multiple economic events within a process, creating a full audit trail of transformations.

In [64]:
# Simplified event creation - produce aluminium bike frame
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note='consume aluminium for bike frame'
amount = 5
cur_pros = process_data['Create_aluminium_bike_frame']
cur_res = res_data['aluminium']

event_id, ts = create_event(users_data['manufacturer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

# Produce aluminum bike frame
action = 'produce'
event_note='produce aluminium bike frame'
amount = 1
res_data['aluminium_bike_frame'] = {
    "res_ref_id": f'aluminium_bike_frame-{random.randint(0, 10000)}',
    "name": 'aluminium bike frame',
    "spec_id": res_spec_data['aluminium_bike_frame']['id']
}
cur_res = res_data['aluminium_bike_frame']

event_id, ts = create_event(users_data['manufacturer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

### 13.1 Produce Aluminium Bike Frame

Manufacturer1 transforms aluminium into a bike frame:
1. **Consume**: 5 kg of aluminium (input to process)
2. **Produce**: 1 aluminium bike frame (output from process)

Both events are part of the "Create_aluminium_bike_frame" process.

In [65]:
# Transfer the aluminium bike frame from manufacturer1 to designer2
cur_res = action = event_note = amount = cur_pros = None
note='Transfer aluminium bike frame from manufacturer1 to designer2'
action = 'transfer'
amount = 1
cur_res = res_data['aluminium_bike_frame']

event_id, ts = make_transfer(users_data['manufacturer1'], action, note, users_data['designer2'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

### 13.2 Transfer Frame to Designer2

The aluminium bike frame is transferred from manufacturer1 to designer2.

This creates a transfer event that:
- Reduces manufacturer1's inventory
- Creates a new resource instance in designer2's custody
- Records the transfer location and timestamp

In [66]:
# Produce magic bike mirror
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note='consume glass for bike mirror'
amount = .2
cur_pros = process_data['Creation_magic_bike_mirror']
cur_res = res_data['glass']

event_id, ts = create_event(users_data['manufacturer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

action = 'produce'
event_note='produce magic bike mirror'
amount = 1
res_data['magic_bike_mirror'] = {
    "res_ref_id": f'magic_bike_mirror-{random.randint(0, 10000)}',
    "name": 'magic bike mirror',
    "spec_id": res_spec_data['magic_bike_mirror']['id']
}
cur_res = res_data['magic_bike_mirror']

event_id, ts = create_event(users_data['manufacturer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

### 13.3 Produce Magic Bike Mirror

Manufacturer2 creates the mirror component:
1. **Consume**: 0.2 kg of glass (input to process)
2. **Produce**: 1 magic bike mirror (output from process)

Both events are part of the "Creation_magic_bike_mirror" process.

In [67]:
# Transfer the mirror from manufacturer2 to designer2
cur_res = action = event_note = amount = cur_pros = None
note='Transfer magic bike mirror from manufacturer2 to designer2'
action = 'transfer'
amount = 1
cur_res = res_data['magic_bike_mirror']

event_id, ts = make_transfer(users_data['manufacturer2'], action, note, users_data['designer2'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

### 13.4 Transfer Mirror to Designer2

The magic bike mirror is transferred from manufacturer2 to designer2.

Designer2 now has both physical components (frame and mirror) needed for bike assembly.

In [68]:
# Create funky bike design
cur_res = action = event_note = amount = cur_pros = None
action = 'work'
event_note='work to create funky bike design'
cur_pros = process_data['Create_funky_bike_design']
effort_spec = {}
effort_spec['unit_id'] = res_spec_data['design_work']['defaultUnit']
effort_spec['spec_id'] = res_spec_data['design_work']['id']
effort_spec['amount'] = 16

event_id, ts = create_event(users_data['designer1'], action, event_note, amount=0, process=cur_pros, \
                 res_spec_data=res_spec_data, effort_spec=effort_spec, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'amount': effort_spec['amount']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

action = 'produce'
event_note='produce design for funky bike'
amount = 1
res_data['funky_bike_design'] = {
    "res_ref_id": f'funky_bike_design-{random.randint(0, 10000)}',
    "name": 'funky bike design',
    "spec_id": res_spec_data['funky_bike_design']['id']
}
cur_res = res_data['funky_bike_design']

event_id, ts = create_event(users_data['designer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

### 13.5 Create Funky Bike Design

Designer1 creates the bike design through intellectual work:
1. **Work**: 16 hours of design effort (input to process)
2. **Produce**: 1 funky bike design (output from process)

The design will be **cited** (not consumed) when the bike is assembled, representing intellectual property use without destruction of the original.

In [69]:
# Create the GC1DPP data for the fancy collaborative bike (BEFORE producing it)
bike_dpp = create_sample_bike_dpp()

# Customize the DPP data with information about the components we've already created
bike_dpp['productOverview']['productName']['value'] = 'fancy collaborative bike'
bike_dpp['productOverview']['productDescription']['value'] = f"A unique collaborative bike"

# Add component information based on the resources we created
bike_dpp['components'] = [
    {
        "componentDescription": {"type": "string", "value": res_data['aluminium_bike_frame']['name']},
        "componentGTIN": {"type": "string", "value": res_data['aluminium_bike_frame']['res_ref_id']}
    },
    {
        "componentDescription": {"type": "string", "value": res_data['magic_bike_mirror']['name']},
        "componentGTIN": {"type": "string", "value": res_data['magic_bike_mirror']['res_ref_id']}
    },
    {
        "componentDescription": {"type": "string", "value": res_data['funky_bike_design']['name']},
        "componentGTIN": {"type": "string", "value": res_data['funky_bike_design']['res_ref_id']},
        "linkToDPP": {"type": "string", "value": "Referenced design"}
    }
]

# Update economic operator with actual designer2 info
bike_dpp['economicOperator']['companyName']['value'] = users_data['designer2']['name']
bike_dpp['economicOperator']['addressLine1']['value'] = locs_data['designer2']['addr'].split(',')[0]
bike_dpp['economicOperator']['addressLine2']['value'] = ','.join(locs_data['designer2']['addr'].split(',')[1:])

print("DPP data created (before producing bike):")
print(json.dumps(bike_dpp, indent=2))

DPP data created (before producing bike):
{
  "productOverview": {
    "brandName": {
      "type": "string",
      "value": "Fancy Collaborative Bikes"
    },
    "productName": {
      "type": "string",
      "value": "fancy collaborative bike"
    },
    "productDescription": {
      "type": "string",
      "value": "A unique collaborative bike"
    },
    "countryOfOrigin": {
      "type": "string",
      "value": "Netherlands"
    },
    "color": {
      "type": "string",
      "value": "Custom"
    },
    "netWeight": {
      "type": "number",
      "value": 15,
      "units": "kg"
    },
    "modelName": {
      "type": "string",
      "value": "FCB-2025"
    }
  },
  "reparability": {
    "availabilityOfSpareParts": {
      "type": "string",
      "value": "Available through manufacturer network"
    }
  },
  "environmentalImpact": {
    "co2eEmissionsPerUnit": {
      "type": "number",
      "value": 45,
      "units": "kg"
    },
    "minimumContentOfMaterialWithSustainabilit

## 14. Create GC1DPP Digital Product Passport (BEFORE Production)

**This is the key innovation**: Create and submit the Digital Product Passport **BEFORE** producing the final bike.

### Steps:
1. Generate a sample GC1DPP data structure
2. Customize it with product information (name, description)
3. Add component traceability:
   - Aluminium bike frame (with tracking ID)
   - Magic bike mirror (with tracking ID)
   - Funky bike design (cited reference)
4. Add economic operator information (designer2's company and address)

The DPP follows the GC1DPP (Global Circularity 1) standard for Digital Product Passports, which includes:
- Product overview and identification
- Component list with traceability
- Sustainability information
- Economic operator details

In [76]:
# Submit the DPP to the DPP service BEFORE creating the bike
try:
    dpp_ulid = submit_dpp(
        bike_dpp,
        users_data['designer2']['eddsa_public_key'],
        users_data['designer2']['keyring']['eddsa'],
        DPP_URL
    )
    
    print(f"✓ DPP submitted successfully with ULID: {dpp_ulid}")
    
    # Store the DPP ULID - we'll use it when creating the bike
    bike_metadata = {
        'dpp': dpp_ulid
    }
    
    print(f"✓ DPP ULID ready to be added to bike metadata: {dpp_ulid}")
    
except Exception as e:
    print(f"✗ Error submitting DPP: {e}")
    print("Note: Make sure the DPP service is running at", DPP_URL)
    dpp_ulid = None
    bike_metadata = {}

Submitting DPP to http://localhost:8080/dpp
Public key: EvVXX8mie1bE2vBuC9Tc...
Signature: AYGkHvP5/YZY0XXBbp1hUne9GB7a1uP8rLb66ceG...
DPP submitted with ULID: 01KAX51YNBD5AM3C0AWJZHYSKQ
✓ DPP submitted successfully with ULID: 01KAX51YNBD5AM3C0AWJZHYSKQ
✓ DPP ULID ready to be added to bike metadata: 01KAX51YNBD5AM3C0AWJZHYSKQ
DPP submitted with ULID: 01KAX51YNBD5AM3C0AWJZHYSKQ
✓ DPP submitted successfully with ULID: 01KAX51YNBD5AM3C0AWJZHYSKQ
✓ DPP ULID ready to be added to bike metadata: 01KAX51YNBD5AM3C0AWJZHYSKQ


## 15. Submit DPP to GC1DPP Service

Submit the Digital Product Passport to the DPP service and receive a ULID (Universally Unique Lexicographically Sortable Identifier).

### Authentication:
- The DPP is cryptographically signed using designer2's EdDSA private key
- The signature is verified against the public key
- This ensures the DPP is authentic and tamper-proof

### Output:
- **dpp_ulid**: A unique identifier for this DPP (e.g., "01KAX45XX1XSACK6PG3W5FW04Z")
- **bike_metadata**: A dictionary containing the DPP ULID that will be embedded in the bike

**This ULID will be included in the bike's metadata when we create it in the next step.**

## 16. Produce the Fancy Collaborative Bike (with DPP Metadata)

Now we produce the final bike, embedding the DPP ULID in its metadata at creation time.

### Process Steps:
1. **Consume** the magic bike mirror (physical component)
2. **Cite** the funky bike design (intellectual property reference)
3. **Consume** the aluminium bike frame (physical component)
4. **Produce** the fancy collaborative bike **WITH METADATA**

### Metadata Structure:
```json
{
  "dpp": "01KAX45XX1XSACK6PG3W5FW04Z"
}
```

The metadata is passed to the `create_event` function and added to the GraphQL mutation as `event.resourceMetadata`. This links the GC1DPP to the Zenflows resource from the moment of creation, enabling full traceability.

This matches the frontend pattern where:
1. DPP is created first
2. DPP ULID is obtained
3. Bike is created with metadata containing the DPP ULID

In [78]:
# Debug: Check what we're about to send
print("bike_metadata content:")
print(bike_metadata)
print("\nType:", type(bike_metadata))
if bike_metadata:
    print("dppUlid:", bike_metadata.get('dpp'))

bike_metadata content:
{'dpp': '01KAX51YNBD5AM3C0AWJZHYSKQ'}

Type: <class 'dict'>
dppUlid: 01KAX51YNBD5AM3C0AWJZHYSKQ


### 16.1 Debug: Verify Metadata Structure

Before creating the bike, verify that the metadata is correctly structured:
- Contains the 'dpp' key
- Has the correct ULID value
- Is a Python dictionary (will be serialized to JSON)

In [ ]:
# Produce the fancy collaborative bike with all consumption/cite events
cur_res = action = event_note = amount = cur_pros = None
cur_pros = process_data['Creation_fancy_collaborative_bike']

# consume mirror for bike
action = 'consume'
event_note='consume mirror for bike'
amount = 1
cur_res = res_data['magic_bike_mirror']

event_id, ts = create_event(users_data['designer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# cite design for bike
action = 'cite'
event_note='cite design to build bike'
amount = 1
cur_res = res_data['funky_bike_design']

event_id, ts = create_event(users_data['designer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# consume aluminium bike frame for bike
action = 'consume'
event_note='consume aluminium bike frame'
amount = 1
cur_res = res_data['aluminium_bike_frame']

event_id, ts = create_event(users_data['designer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

# Produce the final bike WITH DPP METADATA
action = 'produce'
event_note='produce fancy collaborative bike'
amount = 1
res_data['fancy_collaborative_bike'] = {
    "res_ref_id": f'fancy_collaborative_bike-{random.randint(0, 10000)}',
    "name": 'fancy collaborative bike',
    "spec_id": res_spec_data['fancy_collaborative_bike']['id']
}
cur_res = res_data['fancy_collaborative_bike']

# Create the bike with DPP metadata
event_id, ts = create_event(users_data['designer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, metadata=bike_metadata, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Bike created with ID: {cur_res['id']}")
if bike_metadata:
    print(f"✓ Bike metadata includes DPP ULID: {bike_metadata.get('dpp', 'N/A')}")

✓ Bike created with ID: 06DBMMAW03G493D6HJ7NTGGQ74
✓ Bike metadata includes DPP ULID: N/A


### 16.2 Execute Bike Production with DPP Metadata

Perform all economic events to create the bike:

**Input Events** (part of Creation_fancy_collaborative_bike process):
- Consume 1 magic bike mirror
- Cite 1 funky bike design (non-destructive reference)
- Consume 1 aluminium bike frame

**Output Event**:
- Produce 1 fancy collaborative bike
- **With metadata**: `{"dpp": "01KAX45XX1XSACK6PG3W5FW04Z"}`

The `create_event` function with `metadata=bike_metadata` parameter:
1. Serializes the metadata to JSON
2. Adds it to the GraphQL mutation as `event.resourceMetadata`
3. Zenflows stores the metadata with the resource

Result: The bike now has built-in traceability to its GC1DPP record.

In [73]:
# Save the DPP data to file for future reference
if dpp_ulid:
    res_data['fancy_collaborative_bike']['dpp_ulid'] = dpp_ulid
    dpp_data['fancy_collaborative_bike'] = {
        'ulid': dpp_ulid,
        'resource_id': res_data['fancy_collaborative_bike']['id'],
        'resource_ref_id': res_data['fancy_collaborative_bike']['res_ref_id'],
        'dpp': bike_dpp
    }
    
    # Save DPP data to file
    if os.path.isfile(DPP_FILE):
        with open(DPP_FILE, 'r') as f:
            existing_dpp_data = json.loads(f.read())
    else:
        existing_dpp_data = {}
    
    existing_dpp_data.update(dpp_data)
    
    with open(DPP_FILE, 'w') as f:
        json.dump(existing_dpp_data, f, indent=2)
    
    print(f"✓ DPP data saved to {DPP_FILE}")
else:
    print("⚠ No DPP ULID available to save")

✓ DPP data saved to use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/dpp_data.json


## 17. Save DPP Data for Future Reference

Store the DPP data locally for offline access and archival purposes.

The saved data includes:
- **ulid**: The DPP's unique identifier
- **resource_id**: The Zenflows resource ID (bike)
- **resource_ref_id**: The tracking identifier (bike serial number)
- **dpp**: The complete GC1DPP data structure

This creates a local index linking Zenflows resources to their DPP records.

## 18. Understanding the Integration

### How DPP Links to Zenflows

The bike's metadata stored in Zenflows contains:
```json
{
  "dpp": "01KAX45XX1XSACK6PG3W5FW04Z"
}
```

This ULID can be used to:
1. Query the DPP service for the complete Digital Product Passport
2. Verify authenticity via cryptographic signatures
3. Access component traceability and sustainability information

### GraphQL Mutation Structure

The `createEconomicEvent` mutation used:
```graphql
mutation($event: EconomicEventCreateParams!, 
         $newInventoriedResource: EconomicResourceCreateParams) {
  createEconomicEvent(
    event: $event, 
    newInventoriedResource: $newInventoriedResource
  ) {
    economicEvent {
      id
      resourceInventoriedAs {
        id
        name
        metadata
      }
    }
  }
}
```

With variables:
```json
{
  "event": {
    "action": "produce",
    "resourceMetadata": "{\"dpp\": \"01KAX45XX1XSACK6PG3W5FW04Z\"}",
    ...
  },
  "newInventoriedResource": {
    "name": "fancy collaborative bike",
    "trackingIdentifier": "fancy_collaborative_bike-5981"
  }
}
```

### Benefits of This Approach

1. **Immutable Link**: DPP ULID is embedded at creation, can't be changed later
2. **Provenance**: Full supply chain visible through both Zenflows events and DPP components
3. **Compliance**: Meets regulatory requirements for product documentation
4. **Circularity**: Enables end-of-life tracking and material recovery

In [74]:
# Display summary of created resources and DPP
show_data(users_data, locs_data, res_data, units_data, res_spec_data, process_data, event_seq)

Users
{
  "designer1": {
    "userChallenges": {
      "whereParentsMet": "London",
      "nameFirstPet": "Fuffy",
      "nameFirstTeacher": "Jim",
      "whereHomeTown": "Paris",
      "nameMotherMaid": "Wright"
    },
    "name": "Designer1",
    "username": "designer1_username",
    "email": "designer1@example.org",
    "note": "me.designer1.org",
    "seedServerSideShard.HMAC": "0KWRJbyzkfa4OwWO9K9dnmQKSg7pkrxPeYRmdalLerU=",
    "seed": "solution garage know special trap wheel timber raven measure miracle achieve horn",
    "eddsa_public_key": "Dqizx57FCxzNcSHo36pzn35dP9VC4cmi1zLCKthzMGLj",
    "keyring": {
      "eddsa": "6W3UnVWSFFbB5G483ggyAwWbJJEJ5ZuoFRRtsDT4Egxg"
    },
    "id": "06DBM3GNH9ME07AG8XDYR0VF98",
    "location_id": "06DBM3HHKYSM2SRY209NYPDGHM"
  },
  "designer2": {
    "userChallenges": {
      "whereParentsMet": "London",
      "nameFirstPet": "Fido",
      "nameFirstTeacher": "Mary",
      "whereHomeTown": "Amsterdam",
      "nameMotherMaid": "Wraight"
    },
  

## 19. Display Summary Data

Show a comprehensive summary of all created entities:
- Users and their locations
- Resources and their specifications
- Processes and their associated events
- Event sequence with timestamps

This provides a quick overview of the entire workflow.

## 20. Supply Chain Tracing and Visualization

The following cells demonstrate how to trace the complete supply chain backwards from the final product.

### Tracing Capabilities:
1. **Backward Tracing** (er_before): Find all inputs and events that led to the bike
2. **DPP Query**: Retrieve the DPP structure from Zenflows
3. **Frontend Trace**: Match backend data with what the frontend would display
4. **Visualization**: Generate interactive Sankey diagrams of material flows

This shows the complete provenance: from raw materials (aluminium, glass) through components (frame, mirror, design) to the final bike.

In [75]:
trace_me = res_data['fancy_collaborative_bike']['id']
print(f"Resource to be traced: {trace_me}")
tot_dpp = []
visited = set()
er_before(trace_me, users_data['designer2'], dpp_children=tot_dpp, depth=0, visited=visited, endpoint=ENDPOINT)

# Serializing json
json_object = json.dumps(tot_dpp, indent=2)

print(json_object)
print(visited)

Resource to be traced: 06DBMJWSC6HWJ6HC8R1EWBP0WC
Payload
{'query': 'query($id:ID!) {\n        economicEvent(id:$id) {\n            ...event\n            previous{\n                __typename\n                ... on  EconomicResource {\n                    id\n                    name\n                }\n                ... on EconomicEvent {\n                    id\n                    action {\n                        id\n                    }\n                }\n                ... on Process {\n                    id\n                    name\n                }\n            }\n          }\n        }\n    \n    fragment event on EconomicEvent {\n        type : __typename\n        action {\n            ...action\n        }\n        agreedIn\n        # String Reference to an agreement between agents which specifies the rules or policies or calculations which govern this economic event.\n\n        atLocation {\n            ...location\n        }\n\n        effortQuantity {\n           

Exception: Exception Expecting value: line 1 column 1 (char 0) in function send_signed

### 20.1 Backward Tracing: Find All Inputs

Trace backwards from the bike to discover all input resources and events.

The `er_before` function recursively follows:
- **Consume events**: Physical components used
- **Cite events**: Intellectual property referenced
- **Transfer events**: Component custody changes
- **Produce events**: Component creation

Result: A tree structure showing the complete bill of materials and provenance.

In [26]:
be_dpp = get_dpp(trace_me, endpoint=ENDPOINT)
print(json.dumps(be_dpp, indent=2))

[
  {
    "node": {
      "accountingQuantity": {
        "hasNumericalValue": "1",
        "hasUnit": {
          "id": "06DBM3KM9R0RN96E4V8XJ68K3W"
        }
      },
      "classifiedAs": null,
      "conformsTo": {
        "id": "06DBM3ZYA1TC7VEAZ1PH7PW8DC"
      },
      "containedIn": {
        "id": null
      },
      "currentLocation": {
        "id": "06DBM3HP5S7A1A38GXREC5SQE8"
      },
      "custodian": {
        "id": "06DBM3GQK9RBD7MHV4ZPYW7N98"
      },
      "id": "06DBMD22ADZ05HM2BKJNWNHBYR",
      "license": null,
      "licensor": null,
      "lot": {
        "id": null
      },
      "metadata": null,
      "name": "fancy collaborative bike",
      "note": null,
      "okhv": null,
      "onhandQuantityHas": {
        "hasUnit": {
          "id": "06DBM3KM9R0RN96E4V8XJ68K3W"
        },
        "numericalValue": "1"
      },
      "previousEvent": {
        "id": "06DBMD22A2201FMA52R66A120C"
      },
      "primaryAccountable": {
        "id": "06DBM3GQK9RBD7MHV4ZPY

### 20.2 Query DPP from Zenflows

Retrieve the Digital Product Passport structure directly from Zenflows backend.

This shows how the DPP data is stored and can be queried using the resource ID. The backend DPP includes metadata about components and their relationships.

In [27]:
trace = trace_query(trace_me, endpoint=ENDPOINT)
check_traces(trace, event_seq, tot_dpp, be_dpp)

################################################################################
nr trace: 26, nr events: 18, nr front-end dpp: 26, nr back-end dpp: 26
################################################################################
Check whether there are any duplicated trace items
################################################################################
Check whether there are any duplicated events
################################################################################
Check whether there are any duplicated nodes in front-end dpp
################################################################################
Check whether there are any duplicated nodes in back-end dpp
################################################################################
Are trace items in the events?
NOT FOUND: trace item fancy collaborative bike id: 06DBMD22ADZ05HM2BKJNWNHBYR of type EconomicResource
NOT FOUND: trace item aluminium id: 06DBMCZ0Q2V7B67WHQX5ZGDHBC of type EconomicResource
N

### 20.3 Trace Validation

Query the frontend-style trace and compare with backend data.

Validates that:
- All events are correctly recorded
- The trace matches the event sequence we created
- Backend DPP matches the frontend trace
- No data is missing or inconsistent

In [ ]:
save_traces(USE_CASE, tot_dpp, trace, be_dpp, event_seq)

### 20.4 Save Trace Data

Save all trace data to JSON files for archival and analysis:
- Frontend trace (user-facing view)
- Backend trace (complete provenance)
- Event sequence (chronological actions)
- DPP structure (Digital Product Passport)

Files are saved to the `traces/` directory.

In [ ]:
labels = []
sources = []
targets = []
values = []
color_nodes = []
color_links = []
assigned = {}
vis_dpp(tot_dpp[0], count=0, assigned=assigned, labels=labels, targets=targets, sources=sources, values=values, color_nodes=color_nodes, color_links=color_links)
sources, targets = consol_trace(assigned, sources, targets)
make_sankey(sources, targets, labels, values, color_nodes, color_links)

### 20.5 Visualize Supply Chain with Sankey Diagram

Generate an interactive Sankey diagram showing material and component flows.

The visualization shows:
- **Nodes**: Resources, processes, agents
- **Links**: Flows between nodes (consume, produce, transfer)
- **Colors**: Different types of resources and actions
- **Widths**: Quantities (where applicable)

This provides an intuitive visual representation of the complete supply chain from raw materials to final product, making it easy to understand the collaborative nature of the bike production.